# Cryptography Laboratory
## Assignment 10: MT19937 Stream Cipher and Predictable Token Attack

### Part A: MT19937 Stream Cipher

**Objective:**

To understand why a statistical pseudorandom number generator (PRNG)
is unsuitable for cryptographic applications by implementing a toy
stream cipher, recovering a weak seed using brute force, and analyzing
a known-plaintext attack.

**Technology Used:**
- Programming Language: Python
- PRNG: MT19937 (Mersenne Twister)
- Library: random
- IDE: Visual Studio Code

## 1. Introduction

MT19937 is a pseudorandom number generator based on the
Mersenne Twister algorithm.

It is widely used in simulations and statistical applications
because it produces a long sequence of statistically good
pseudorandom numbers.

However, MT19937 is not cryptographically secure because
its outputs are predictable under certain conditions.

In this assignment, MT19937 is used to generate keystream
bytes. These bytes are XORed with plaintext to implement
a simple stream cipher.

## 2. Theory

### What is MT19937?

MT19937 is a pseudorandom number generator with:
- A period of 2^19937 - 1.
- A 624-word internal state.
- 32-bit output values.

Python's built-in random module uses MT19937.

### What is a Stream Cipher?

A stream cipher encrypts data one byte at a time using
a generated keystream.

Encryption:

C[i] = P[i] XOR K[i]

Decryption:

P[i] = C[i] XOR K[i]

Where:
- P[i] = Plaintext byte
- K[i] = Keystream byte
- C[i] = Ciphertext byte

XOR is reversible, so the same operation is used
for both encryption and decryption.

### Why is MT19937 not cryptographically secure?

1. Its output is deterministic for a given seed.
2. Its internal state can be reconstructed from sufficient
   output observations.
3. It is not designed to resist cryptographic attacks.
4. Small or predictable seeds make brute-force attacks easier.

In [1]:
import random
import string

In [2]:
def stream_cipher(data, seed):

    rng = random.Random(seed)

    result = bytearray()

    for byte in data:
        key_byte = rng.getrandbits(8)
        result.append(byte ^ key_byte)

    return bytes(result)

In [3]:
message = "Hello Cryptography Lab!"

plaintext = message.encode()
seed = 12345

ciphertext = stream_cipher(plaintext, seed)

decrypted = stream_cipher(ciphertext, seed)

print("Original Message:", message)
print("Seed:", seed)
print("Ciphertext:", ciphertext.hex())
print("Decrypted Message:", decrypted.decode())

print("Verification:", plaintext == decrypted)

Original Message: Hello Cryptography Lab!
Seed: 12345
Ciphertext: 22de6ebdbced0fa8279e452af71d482f77a64e0eeec20d
Decrypted Message: Hello Cryptography Lab!
Verification: True


## 3. Known Plaintext Generation

A random printable prefix of length 5 to 20 characters
is generated.

After the prefix, 14 consecutive 'A' characters are appended.

The plaintext is encrypted using a randomly selected
16-bit seed.

The attacker knows that the plaintext contains 14 consecutive
'A' characters but does not know their position.

In [4]:
prefix_length = random.randint(5, 20)

prefix = ''.join(
    random.choice(string.ascii_letters + string.digits)
    for _ in range(prefix_length)
)

plaintext_message = prefix + "A" * 14
plaintext = plaintext_message.encode()

seed = random.randint(0, 65535)

ciphertext = stream_cipher(plaintext, seed)

print("Generated Prefix:", prefix)
print("Plaintext:", plaintext_message)
print("Plaintext Length:", len(plaintext))
print("Random 16-bit Seed:", seed)
print("Ciphertext (Hex):", ciphertext.hex())

Generated Prefix: MQJnATaxVyLnwVM227D
Plaintext: MQJnATaxVyLnwVM227DAAAAAAAAAAAAAA
Plaintext Length: 33
Random 16-bit Seed: 57418
Ciphertext (Hex): 29c8f42d484f982caf6b629a094c92415f6dd4cbd0bd698a811c798fadc60df252


## 4. Brute-Force Seed Recovery

The seed is a 16-bit integer.

Therefore, the total number of possible seeds is:

2^16 = 65536

The attacker tests every possible seed from 0 to 65535.

For each candidate seed:
1. Generate the keystream.
2. Decrypt the ciphertext.
3. Search for 14 consecutive 'A' characters.
4. If the pattern is found, consider the candidate seed.

The matching seed is then used to decrypt the complete ciphertext.

In [5]:
def recover_seed(ciphertext):

    known_pattern = b"A" * 14

    candidates_tested = 0

    for candidate_seed in range(65536):

        candidates_tested += 1

        decrypted = stream_cipher(ciphertext, candidate_seed)

        if known_pattern in decrypted:

            return candidate_seed, decrypted, candidates_tested

    return None, None, candidates_tested

In [6]:
recovered_seed, recovered_plaintext, tested = recover_seed(ciphertext)

print("Total Candidates Tested:", tested)

if recovered_seed is not None:

    print("Original Seed:", seed)
    print("Recovered Seed:", recovered_seed)

    print("Recovered Plaintext:", recovered_plaintext.decode())

    print("Seed Matches:", recovered_seed == seed)
    print("Plaintext Matches:", recovered_plaintext == plaintext)
    print("Known Pattern Found:", b"A" * 14 in recovered_plaintext)

else:
    print("No matching seed found.")

Total Candidates Tested: 57419
Original Seed: 57418
Recovered Seed: 57418
Recovered Plaintext: MQJnATaxVyLnwVM227DAAAAAAAAAAAAAA
Seed Matches: True
Plaintext Matches: True
Known Pattern Found: True


## 5. Observations

1. MT19937 successfully generated pseudorandom keystream bytes.
2. XOR encryption and decryption produced the original plaintext.
3. A random 16-bit seed was selected for encryption.
4. The known plaintext consisted of 14 consecutive 'A' characters.
5. Brute-force search tested candidate seeds in the range 0 to 65535.
6. The correct seed was recovered by identifying the known plaintext pattern.
7. The recovered seed successfully decrypted the complete ciphertext.

## 6. Conclusion

This experiment demonstrates that MT19937 is unsuitable
for cryptographic applications.

A small seed search space makes brute-force seed recovery
practical when the attacker has useful known-plaintext
information.

Cryptographic applications should use a
cryptographically secure pseudorandom number generator
(CSPRNG) instead of a statistical PRNG.